# Chapitre 12 - Lab : dégrouper la base avec PostgreSQL et Redis

Les cours de ce chapitre défendent une idée : une organisation peut se construire comme
une **base de données dégroupée** (« unbundled »), où un système de référence publie le
journal ordonné de ses changements et où index, caches et compteurs sont des **vues
dérivées**, entretenues par des consommateurs idempotents et recalculables à volonté. Ce
lab met l'idée à l'épreuve sur de vrais systèmes : **PostgreSQL 17** joue le système de
référence, son **décodage logique** (plugin `test_decoding`) fournit le flux de
changements, et **Redis 7** héberge un index inversé et des compteurs dérivés. On y
provoque des pannes de consommateur, un bogue dans la fonction de dérivation, des
relances de requêtes et des courses sur une contrainte d'unicité.

**Démarrer et arrêter les services** (depuis la racine du dépôt) :

```bash
docker compose -f docker/dataflow/compose.yml -p ddia_dataflow up -d
# ... exécuter le notebook ...
docker compose -f docker/dataflow/compose.yml -p ddia_dataflow down -v
```

PostgreSQL écoute sur le port **11332** de l'hôte (utilisateur, mot de passe et base :
`ddia`), lancé avec `wal_level=logical` ; Redis écoute sur le port **11379**.

**Plan.**

1. Le journal de changements du système de référence.
2. Entretenir des vues dérivées dans Redis, et mesurer leur retard.
3. Panne du consommateur : relivraison et idempotence.
4. `CREATE INDEX CONCURRENTLY` : une vue dérivée construite par instantané et rattrapage.
5. Faire confiance, mais vérifier : audit de bout en bout d'une vue dérivée.
6. Corriger par retraitement : dériver une nouvelle vue à côté de l'ancienne.
7. Identifiants de requête de bout en bout.
8. Contraintes d'unicité sous concurrence.

In [1]:
import random
import threading
import time
import uuid
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg
import redis
from psycopg import errors

from ddia.dataflow_pg import InvertedIndexView, parse_change, rebuild_index, tokenize
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, finish, set_style, subtitle)
from ddia.timelines import SpaceTime

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.max_colwidth", 90)
DSN = "host=localhost port=11332 user=ddia password=ddia dbname=ddia connect_timeout=5"
R = redis.Redis(host="localhost", port=11379, decode_responses=True)

with psycopg.connect(DSN, autocommit=True) as c:
    print(c.execute("SELECT version()").fetchone()[0].split(",")[0])
    print("wal_level =", c.execute("SHOW wal_level").fetchone()[0])
print("Redis", R.info("server")["redis_version"], "ping :", R.ping())


def sql(q, params=None):
    """Exécute une requête dans une connexion autocommit et renvoie les lignes."""
    with psycopg.connect(DSN, autocommit=True) as c:
        cur = c.execute(q, params)
        return cur.fetchall() if cur.description else None


# remise à zéro : le notebook peut être rejoué
for (slot,) in sql("SELECT slot_name FROM pg_replication_slots"):
    sql("SELECT pg_drop_replication_slot(%s)", (slot,))
sql("DROP TABLE IF EXISTS doc, mesures, comptes, requetes, journal_effets, "
    "utilisateurs, utilisateurs_u, demandes_noms CASCADE")
R.flushdb()

PostgreSQL 17.11 on x86_64-pc-linux-musl
wal_level = logical
Redis 7.4.11 ping : True


True